## Earthquakes In Italy

Network 10x10x10km

Here we try an hybrid version between hard version (with hard threshold for spatial/temporal distances between consecutive earthquakes) and soft version (that keeps all links but weights them exponentially using spatial/temporal distance and magnitude of the two consecutive earthquakes).

In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
Abe-Suzuki Earthquake Network — Italy INGV Catalog (1985-2025)

"""

import logging
import time
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from shapely.geometry import Point, Polygon
import plotly.express as px
import plotly.io as pio
import seaborn as sns
from sklearn.metrics import normalized_mutual_info_score

from src.network import (
    build_abe_suzuki_network,
    discretize_space_3d
)

from src.network_custom import build_abe_suzuki_network_custom_hybrid


from src.metrics import (
    fit_strength_distribution_hybrid,
    fit_lognormal_full_hybrid,
    test_power_law,
    verify_balanced_degrees,
)
from src.assortativity_custom import (
    analyze_degree_correlations_hybrid,
    run_binned_randomization_test_hybrid,
    preprocess_to_binned_df,
    fit_intrinsic_slope
)
from src.robustness import simulate_robustness, plot_robustness_curves

from src.centrality_custom import (
    compute_all_centralities_hybrid,
    plot_centrality_correlation_hybrid,
    plot_geo_top_n_interactive_hybrid,
    plot_geo_centrality_overlap_hybrid
)
from src.community_custom import (
    _to_igraph,
    _run_leiden,
    run_louvain_hybrid,
    run_louvain_directed_hybrid,
    run_louvain_consensus_hybrid,
    run_infomap_hybrid,
    run_mmsbm_custom_hybrid,
    run_hdbscan_geo_hybrid,
    plot_community_geo_hybrid,
    align_partitions,
    compute_nmi,
    compute_nmi_matrix,
    plot_nmi_heatmap
)
from src.viz import (
    analyze_degree_distribution_hybrid,
    analyze_in_out_degree_distribution_hybrid,
    analyze_strength_distribution_log_binning_hybrid,
    plot_ccdf_with_fit_hybrid
)
from src.plotutils import setup_matplotlib, configure_saves, savefig, save_plotly

try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore[assignment]

logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
# Silence chatty third-party loggers (Kaleido/Chromium PDF export spam, matplotlib
# font cache, PIL image debug, asyncio cleanup) — keep our own INFO messages visible.
for _noisy in ("kaleido", "choreographer", "logistro", "matplotlib",
               "PIL", "urllib3", "asyncio"):
    logging.getLogger(_noisy).setLevel(logging.WARNING)
sns.set_theme(style="whitegrid")
pio.renderers.default = "notebook"

DATA_DIR    = Path("data/INGV")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
(RESULTS_DIR / "data").mkdir(exist_ok=True)
(RESULTS_DIR / "gephi").mkdir(exist_ok=True)
CUT_YEAR    = 1985
TARGET_CRS  = "epsg:32632"  # UTM Zone 32N — appropriate for Italy
SAVE_PDF: bool = True    # save vector PDF for report
SAVE_JPG: bool = True    # save raster JPG for slides/preview

_IT_BOUNDS = dict(west=3, east=22, south=34, north=48)
MAP_WIDTH  = 770
MAP_HEIGHT = 700

setup_matplotlib()
configure_saves(SAVE_JPG, SAVE_PDF, RESULTS_DIR / "figures" / "italy" / "abe")

## Data Loading

In [ ]:
print("Loading Italy earthquake catalog...")
df = pd.read_csv(DATA_DIR / "italy_earthquakes_1985_2025.csv")
df["time"] = pd.to_datetime(df["time"], utc=True)
df_net = (
    df[df["time"].dt.year >= CUT_YEAR]
    .sort_values("time")
    .reset_index(drop=True)
)

print(f"Loaded {len(df_net):,} earthquakes "
      f"({df_net['time'].dt.year.min()}–{df_net['time'].dt.year.max()})")
print(f"RAM: {df_net.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
display(pd.concat([df_net.head(3), df_net.tail(3)]))

_ITALY_POLYGON = Polygon([
    (13.5, 44.5),  # Northeast (Adriatic Sea)
    (19.0, 41.2),  # Southeast (Apulia Sea)
    (19.0, 35.6),  # Sea between Italy and Libya
    (11.7, 35.5),  # Sea east of Tunisia
    (11.6, 37.9),  # West of Sicily
    (7.2,  38.0),  # Southwest of Sardinia
    (7.2,  42.6),  # West of Corsica
    (5.2,  45.5),  # Lyon
    (11.6, 48.4),  # Munich
    (16.0, 46.7),  # Between Austria and Slovenia
    (13.5, 44.5),  # closing vertex
])
df_net = df_net[df_net.apply(
    lambda r: _ITALY_POLYGON.contains(Point(r["longitude"], r["latitude"])), axis=1
)].reset_index(drop=True)
print(f"After polygon filter: {len(df_net):,} earthquakes remain")

# df_net = df_net[
#     (df_net["magnitude"] > 2.0) &
#     (df_net["magnitude"] <= 10.0) &
#     (df_net["time"].dt.year >= 2005) &
#     (df_net["time"].dt.year <= 2015)
# ].reset_index(drop=True)
# print(f"After magnitude [2–10] and year [2005–2015] filter: {len(df_net):,} earthquakes remain")

neg = df_net[df_net["depth_km"] < 0]
print(f"\nNegative-depth events: {len(neg)} — kept, collapse to cell_z = -1")

## Network Construction

In [ ]:
G_10km = build_abe_suzuki_network_custom_hybrid(
    df_net, 
    cell_size_km=20, 
    spatial_threshold_km = 300.0,   # filter on spatial distance between two consecutive EQ
    time_threshold_sec = 24 * 3600, # filter on time elapsed between two consecutive EQ (in seconds)
    target_crs=TARGET_CRS, 
    alpha = 0.7,            # magnitude scaling
    tau = 1 * 86400.0,      # temporal scale (seconds) ~ for exponential
    r0 = 20.0,              # spatial scale (km) ~ for exponential
    info=True
)

print(f"\n10 km network: {G_10km.number_of_nodes():,} nodes  {G_10km.number_of_edges():,} edges")

weights = [d["weight"] for _, _, d in G_10km.edges(data=True)]

print("===== WEIGHTS ====")
print(f"Min: {np.min(weights):.6f}")
print(f"Max: {np.max(weights):.1f}")
print(f"Mean: {np.mean(weights):.1f}")
print(f"Median: {np.median(weights):.3f}")

# Top 10 strongest edges
top_edges = sorted(G_10km.edges(data=True), key=lambda x: x[2]["weight"], reverse=True)[:3]

# Bottom 10 weakest edges
bottom_edges = sorted(G_10km.edges(data=True), key=lambda x: x[2]["weight"])[:3]

print("Top edges:")
for u, v, d in top_edges:
    print(u, "->", v, d["weight"])

print("\nWeakest edges:")
for u, v, d in bottom_edges:
    print(u, "->", v, d["weight"])

# distrib
plt.hist(np.log10(weights), bins=100)
plt.xlabel("log10(weight)")
plt.ylabel("Frequency")
plt.title("Log-weight distribution")
plt.show()

_IT_BOUNDS = dict(west=3, east=22, south=34, north=48)
MAP_WIDTH  = 770
MAP_HEIGHT = 700

## Hub Map — 2D

The top-2% highest-degree cells are mapped geographically to locate the
most seismically active spatial clusters. Degree here is total unweighted
degree, so hubs correspond to cells visited most frequently in the
earthquake sequence — the busiest fault segments.

In [ ]:
df_nodes = pd.DataFrame([
    {
        "cell_id": n,
        "degree":  G_10km.degree(n),
        "lat":     G_10km.nodes[n]["lat"],
        "lon":     G_10km.nodes[n]["lon"],
    }
    for n in G_10km.nodes() if "lat" in G_10km.nodes[n]
])
df_nodes  = df_nodes[df_nodes["degree"] > 0]
threshold = df_nodes["degree"].quantile(0.98)
df_hubs   = df_nodes[df_nodes["degree"] >= threshold].copy()
print(f"Mapping top 2% hubs: {len(df_hubs)} cells (degree >= {threshold:.0f})")

_IT_BOUNDS = dict(west=3, east=22, south=34, north=48)
MAP_WIDTH  = 770
MAP_HEIGHT = 700

fig = px.scatter_map(
    df_hubs, lat="lat", lon="lon",
    color="degree", size="degree",
    color_continuous_scale="plasma",
    map_style="carto-positron",
    hover_name="cell_id",
    title="Abe-Suzuki Network Hubs — Italy 10×10×10 km (top 2% by degree)",
)
fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0}, width=MAP_WIDTH, height=MAP_HEIGHT,
                  coloraxis_colorbar=dict(title="Degree"),
                  map=dict(center={"lat": 41.9, "lon": 12.5}, zoom=0, bounds=_IT_BOUNDS))
save_plotly(fig, "hub_map_2d_italy_10km")
fig.show()

## Network Node Map — Degree & Depth

Every network node (10×10×10 km spatial cell) is placed on an Italy basemap.
**Marker size** encodes total degree — the number of distinct transition events
entering or leaving the cell — so the largest circles are the most seismically
active cells in the sequence. **Colour** encodes cell-centre depth (km), from
shallow crustal seismicity along the Apennine belt (yellow) to deeper
subduction-related seismicity beneath the Tyrrhenian back-arc (purple).
Together, size and colour reveal that most network hubs are shallow
(seismogenic crust, 5–20 km), while deep cells are relatively isolated —
consistent with the Chiarabba *et al.* (2005) Italy seismicity model.

In [ ]:
_IT_BOUNDS = dict(west=3, east=22, south=34, north=48)
MAP_WIDTH  = 770
MAP_HEIGHT = 700

df_nodes = pd.DataFrame([
    {
        "cell_id":  n,
        "degree":   G_10km.degree(n),
        "lat":      G_10km.nodes[n]["lat"],
        "lon":      G_10km.nodes[n]["lon"],
        "depth_km": (int(n.split("_")[2]) + 0.5) * 10.0,
    }
    for n in G_10km.nodes() if "lat" in G_10km.nodes[n]
])
df_nodes = df_nodes[df_nodes["degree"] > 0].copy()
df_nodes["depth_km"] = df_nodes["depth_km"].clip(upper=200)
df_nodes = df_nodes.sort_values("depth_km", ascending=False)  # deep-first → shallow on top

print(f"Node map: {len(df_nodes):,} active cells  "
      f"degree range [{df_nodes['degree'].min()}–{df_nodes['degree'].max()}]  "
      f"depth range [{df_nodes['depth_km'].min():.0f}–{df_nodes['depth_km'].max():.0f}] km")

fig = px.scatter_map(
    df_nodes,
    lat="lat", lon="lon",
    size="degree",
    color="depth_km",
    color_continuous_scale="plasma_r",
    size_max=30,
    map_style="carto-positron",
    hover_name="cell_id",
    hover_data={"degree": True, "depth_km": ":.1f", "lat": ":.3f", "lon": ":.3f"},
    labels={"depth_km": "Depth (km)", "degree": "Degree"},
    title="Abe-Suzuki Network — Italy 10×10×10 km (size = degree, colour = depth)",
)
fig.update_layout(
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    width=MAP_WIDTH, height=MAP_HEIGHT,
    coloraxis_colorbar=dict(title="Depth (km)"),
    map=dict(center={"lat": 41.9, "lon": 12.5}, zoom=0, bounds=_IT_BOUNDS),
)
save_plotly(fig, "node_map_degree_depth_italy_10km")
fig.show()

## Degree Distribution

## MLE Gamma Estimation

The maximum-likelihood estimator for the power-law exponent (Clauset et al. 2009):
$$\hat{\gamma} = 1 + n\left[\sum_{i=1}^{n}\ln\frac{k_i}{k_{\min}}\right]^{-1}$$
where $n$ is the number of degrees $\geq k_{\min}$ and $k_{\min} = 10$.

The Clauset-Shalizi-Newman (CSN) likelihood-ratio test compares the power-law
to an exponential alternative: $R > 0$ with $p < 0.05$ rejects the exponential
and supports the power-law. Abe & Suzuki (2004) found $\gamma \approx 2.2$
for the Japan catalog; $\gamma < 3$ implies a divergent second moment and
enhanced vulnerability to targeted attack.

The MLE $\hat{\gamma}$ is used as the fixed slope in the log-binned and CCDF
plots below, with only the amplitude fitted to the data — giving a visually
honest overlay that does not double-count estimation error. ``xmin`` is
auto-selected by the powerlaw library via KS minimisation (Clauset *et al.*
2009), not fixed by hand — a tiny user-supplied ``k_min`` would force the
fit to span the bulk of a lognormal-looking weight distribution and yield a
degenerate $\hat{\gamma} \approx 1$.

The hybrid weights are a product of three exponentially-decaying factors
(magnitude, time, space), so by the multiplicative CLT the strength
distribution is approximately **lognormal**. A power-law fit will succeed
numerically on any heavy-tailed data, so we run both fits on the same
auto-selected tail and compare them with Vuong's likelihood-ratio test:
$R = \log\!\big(\mathcal{L}_{\text{PL}} / \mathcal{L}_{\text{LN}}\big)$,
normalised by its standard deviation, with significance $p$. Only when
$R>0$ and $p<0.1$ can scale-freeness be claimed; otherwise lognormal is
preferred or the data does not separate the two.

In [ ]:
strengths_10km = [s for _, s in G_10km.degree(weight="weight") if s > 0]

fit_10km = fit_strength_distribution_hybrid(strengths_10km, k_min=None)
gamma_10km = fit_10km["power_law"]["gamma"]
xmin_10km  = fit_10km["power_law"]["xmin"]
mu_tail    = fit_10km["lognormal"]["mu"]
sigma_tail = fit_10km["lognormal"]["sigma"]
R_10km     = fit_10km["comparison"]["R"]
p_10km     = fit_10km["comparison"]["p"]
preferred_10km = fit_10km["comparison"]["preferred"]

# Full-data lognormal fit (no tail truncation) — interpretable μ, σ for
# describing the bulk shape of the strength distribution.
mu_full, sigma_full = fit_lognormal_full_hybrid(strengths_10km)
median_full = float(np.exp(mu_full))

print("=== Power-law fit (tail only) ===")
print(f"  γ = {gamma_10km:.3f}   (auto-xmin = {xmin_10km:.3g})")
print()
print("=== Lognormal fit ===")
print(f"  full data : μ = {mu_full:+.3f}, σ = {sigma_full:.3f}   "
      f"(median strength = {median_full:.3g})")
print(f"  tail only : μ = {mu_tail:+.3f}, σ = {sigma_tail:.3f}   "
      "(stretched left to match tail — not interpretable)")
print()
print("=== Vuong likelihood-ratio test (on tail s ≥ xmin) ===")
print(f"  R = {R_10km:+.3f}, p = {p_10km:.3g}  → preferred: {preferred_10km}")

## Degree Distribution — Log Binning

Logarithmic binning reduces noise in the sparse high-$k$ tail by grouping
strengths into exponentially spaced bins and normalising by bin width.
This is the standard method recommended by Clauset et al. (2009) for
visually assessing power-law fit quality. The overlaid line uses the MLE
exponent $\hat{\gamma}$ on the auto-selected tail ($s \geq x_{\min}$).

In [ ]:
analyze_strength_distribution_log_binning_hybrid(
    G_10km, "Abe-Suzuki Italy (10x10x10 km)",
    k_min_fit=xmin_10km, gamma_mle=gamma_10km,
)

## Degree Distribution — CCDF

The complementary cumulative distribution function $P(K \geq k)$ avoids
binning artefacts entirely and is monotone non-increasing by construction.
For a power-law $P(k) \propto k^{-\gamma}$, the CCDF follows
$P(K \geq k) \propto k^{-(\gamma-1)}$; the overlaid line uses the MLE
exponent so the slope is $-(\hat{\gamma}-1)$.

In [ ]:
plot_ccdf_with_fit_hybrid(G_10km, "Abe-Suzuki Italy 10 km",
                          k_min_fit=xmin_10km, gamma_mle=gamma_10km)

## Strength Distribution — Lognormal Overlay

Log-binned strength PDF with both candidate fits overlaid on the same axes
for direct visual comparison:

- **Lognormal (full data)**:
$p(s) = \frac{1}{s\,\sigma\sqrt{2\pi}}\exp\!\left[-\frac{(\ln s - \mu)^2}{2\sigma^2}\right]$,
with $\mu, \sigma$ estimated by closed-form MLE on $\ln s$ across all
nodes — no tail truncation. Mechanistically motivated: the hybrid edge
weights are a product of three exponential decays (magnitude, time, space),
so by the multiplicative CLT the strengths are approximately lognormal.

- **Power-law (tail only)**: $p(s) \propto s^{-\hat{\gamma}}$ for
$s \geq x_{\min}$ (auto-selected by KS minimisation), with amplitude
continuous-matched to the lognormal at $x_{\min}$.

Where the two curves overlap (the upper tail) they are visually nearly
identical — this is the well-known wide-lognormal/power-law identification
problem (Mitzenmacher 2004; Clauset *et al.* 2009 §6.2): a lognormal with
large $\sigma$ has an upper tail indistinguishable from a power law unless
the sample contains $\gtrsim 10^5$ points above $x_{\min}$.

The "lognormal body + Pareto tail" structure visible here is captured
natively by the **double Pareto-Lognormal distribution** (Reed & Jorgensen
2004), derived as the stationary distribution of a stopped geometric
Brownian motion — exactly the multiplicative-growth mechanism that produced
the hybrid edge weights. A formal dPLN fit (single likelihood, no arbitrary
$x_{\min}$ cutoff) is left as future work; the qualitative scientific
conclusion — lognormal bulk, heavier upper tail from a small number of
mainshock-sequence cells — is already supported by the plot above.

In [ ]:
strengths_arr = np.asarray([s for _, s in G_10km.degree(weight="weight") if s > 0])

# Log-binned empirical PDF
n_bins = 30
bins = np.logspace(np.log10(strengths_arr.min()), np.log10(strengths_arr.max()), n_bins)
hist, edges = np.histogram(strengths_arr, bins=bins, density=True)
bin_centers = np.sqrt(edges[:-1] * edges[1:])

# Lognormal PDF (natural-log parameterisation; matches fit_lognormal_full_hybrid)
s_grid = np.logspace(np.log10(strengths_arr.min()), np.log10(strengths_arr.max()), 400)
lognormal_pdf = (1.0 / (s_grid * sigma_full * np.sqrt(2.0 * np.pi))
                 ) * np.exp(-(np.log(s_grid) - mu_full) ** 2 / (2.0 * sigma_full ** 2))

# Power-law PDF on tail, amplitude pinned to lognormal value at xmin so the
# two curves meet continuously and the eye can compare their tail slopes.
A_ln_at_xmin = (1.0 / (xmin_10km * sigma_full * np.sqrt(2.0 * np.pi))
                ) * np.exp(-(np.log(xmin_10km) - mu_full) ** 2 / (2.0 * sigma_full ** 2))
s_tail = s_grid[s_grid >= xmin_10km]
powerlaw_pdf = A_ln_at_xmin * (s_tail / xmin_10km) ** (-gamma_10km)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(bin_centers[hist > 0], hist[hist > 0],
           color="purple", s=45, edgecolor="black", zorder=3, label="Log-binned strength")
ax.plot(s_grid, lognormal_pdf,
        color="seagreen", lw=2.2, ls="-",
        label=rf"Lognormal (full): $\mu={mu_full:.2f}$, $\sigma={sigma_full:.2f}$")
ax.plot(s_tail, powerlaw_pdf,
        color="crimson", lw=2.2, ls="--",
        label=rf"Power-law ($s \geq {xmin_10km:.0f}$): $\hat{{\gamma}}={gamma_10km:.2f}$")
ax.axvline(xmin_10km, color="grey", ls=":", lw=1, alpha=0.7)
ax.text(xmin_10km, ax.get_ylim()[1] * 0.5, f"  $x_{{\\min}}={xmin_10km:.0f}$",
        rotation=90, va="top", ha="left", color="grey", fontsize=9)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Strength $s$")
ax.set_ylabel("Probability density $p(s)$")
ax.set_title(f"Strength distribution — Abe-Suzuki Italy (10 km hybrid)\n"
             f"Vuong test on tail: $R={R_10km:+.2f}$, $p={p_10km:.2f}$ → {preferred_10km}")
ax.legend(loc="lower left", fontsize=9)
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
savefig("strength_distribution_lognormal_vs_powerlaw")
plt.show()

## Macroscopic Metrics

The small-world signature (Watts & Strogatz 1998) requires simultaneously
$C_{\text{real}} \gg C_{\text{ER}}$ (high local clustering) and
$L_{\text{real}} \approx L_{\text{ER}}$ (short average path length),
where $C_{\text{ER}} \approx \langle k \rangle / N$ and $L_{\text{ER}} \approx \ln N / \ln \langle k \rangle$.

**Giant component fraction** measures network integrity: values $> 0.9$ indicate
a well-connected seismic system; a fragmented network would imply spatially
isolated fault clusters with no topological path between them.
The adjacency matrix sparsity plot visually confirms the ultra-sparse structure
characteristic of real-world networks with $\langle k \rangle \ll N$.

In [ ]:
G = G_10km

print("--- Giant Component ---")
wcc = list(nx.weakly_connected_components(G))
G_giant = G.subgraph(max(wcc, key=len)).copy()

pct = G_giant.number_of_nodes() / G.number_of_nodes() * 100

print(f"Components: {len(wcc)} | Giant: {G_giant.number_of_nodes():,} nodes ({pct:.1f}%)")

print("\n--- Clustering Coefficient ---")

G_und = G_giant.to_undirected()
G_und.remove_edges_from(nx.selfloop_edges(G_und))

avg_c = nx.average_clustering(G_und)

# ER baseline (same structure assumption as before)
p_rnd = (2 * G_und.number_of_edges() /
         (G_giant.number_of_nodes() * (G_giant.number_of_nodes() - 1)))

print(f"Avg weighted clustering C = {avg_c:.8f}  (Erdos-Renyi p ≈ {p_rnd:.4f})")

print("\n--- Average Path Length & Diameter ---")

t0 = time.time()

# WARNING: uses unweighted shortest paths (still standard in network science)
avg_L = nx.average_shortest_path_length(G_und)
diameter = nx.diameter(G_und)

n_giant = G_giant.number_of_nodes()
mean_k = 2 * G_und.number_of_edges() / n_giant

L_er = np.log(n_giant) / np.log(mean_k)

print(
    f"L_real = {avg_L:.2f}  |  "
    f"L_ER ≈ ln N / ln ⟨k⟩ = {L_er:.2f}  |  "
    f"Diameter = {diameter}  ({time.time()-t0:.0f}s)"
)

print("\n--- Adjacency Matrix Sparsity ---")

adj = nx.to_scipy_sparse_array(G, weight="weight")

sparsity = 1.0 - adj.nnz / (G.number_of_nodes() ** 2)

print(
    f"Shape: {adj.shape}  |  Non-zeros: {adj.nnz:,}  |  "
    f"Sparsity: {sparsity*100:.4f}%"
)

plt.figure(figsize=(8, 8))
plt.spy(adj, markersize=0.05, color="darkblue")
plt.title("Weighted Adjacency Matrix (hybrid network)", fontsize=14)

plt.xlabel("Target Cell Index", fontsize=12)
plt.ylabel("Source Cell Index", fontsize=12)

plt.legend(handles=[
    mpatches.Patch(facecolor="darkblue", label="Weighted link (w > 0)"),
    mpatches.Patch(facecolor="white", edgecolor="lightgray", label="No link"),
], loc="upper right", fontsize=9, framealpha=0.8)

plt.tight_layout()
savefig("adjacency_matrix_hybrid_italy_10km")
plt.show()

## Centrality

## Centrality Analysis — 5 Measures

Five complementary centrality measures capture distinct structural roles in
the Abe–Suzuki seismic cell network.  Each measure encodes a different aspect of
network influence; comparing them reveals whether prominent nodes are prominent
for the same reason (redundancy) or for different reasons (orthogonality).

### Degree

**Total degree** $k_i = k^{\mathrm{in}}_i + k^{\mathrm{out}}_i$ identifies the
most globally active cells, regardless of the direction of influence.

### Path-based measures

**PageRank** (Page *et al.* 1998) solves the stationary equation
$\mathbf{r} = \alpha A^T D^{-1} \mathbf{r} + (1-\alpha)\mathbf{1}/N$
(damping factor $\alpha = 0.85$).  High PageRank indicates a *stress sink*: a cell
that persistently receives seismic flow from well-connected predecessors, not merely
from many predecessors.

**Closeness centrality** $C_i = (N-1)/\sum_j d_{ij}$ is the inverse mean
shortest-path distance.  High-closeness cells can propagate stress signals fastest
across the network — seismological analogue of rapid aftershock diffusion.

**Betweenness centrality** $B_i = \sum_{s \neq i \neq t} \sigma_{st}(i)/\sigma_{st}$
(Freeman 1977) counts the fraction of shortest paths that pass through $i$.
High-betweenness cells act as *seismic bridges* between distinct fault clusters;
removing them would fragment the network into isolated communities.

### Local topology

**Clustering coefficient** $C_i = \frac{2 t_i}{k_i(k_i-1)}$ (Watts & Strogatz 1998)
measures the probability that two neighbours of cell $i$ are also neighbours of each
other.  High clustering signals fault junctions or dense aftershock clusters where
multiple fault strands intersect in a small spatial cell.

### Interpretation of the correlation heatmap

Pairs of measures with Spearman $\rho > 0.9$ are functionally redundant and can
be represented by a single measure in downstream analysis.  Pairs with
$\rho < 0.3$ are structurally orthogonal and reveal genuinely different node roles.
Expected orthogonalities: Betweenness vs Degree (bridges may have low degree),
Clustering vs Degree (peripheral low-degree junctions can have high clustering)

In [ ]:
print("Computing centrality measures (hybrid 10 km network)...")

df_centrality = compute_all_centralities_hybrid(
    G_10km,
    k_betweenness=1000,
    seed=42
)

for metric, label in [
    ("Degree",      "Most Active Cells (Strength)"),
    ("PageRank",    "Stress Sinks"),
    ("Closeness",   "Topological Centers"),
    ("Betweenness", "Fault Bridges"),
    ("Clustering",  "Fault Junctions"),
]:
    print(f"\n--- Top 5: {label} ({metric}) ---")
    display(
        df_centrality.nlargest(5, metric)[
            ["cell_id", "lat", "lon", "depth_km", metric]
        ]
    )

plot_centrality_correlation_hybrid(df_centrality, "Abe-Suzuki Italy (10 km)")

# plot_top_n_cells(df_centrality, top_n=10, title="Abe-Suzuki Italy (10 km)") # to be modified

## Centrality Geo Maps

Geographic projection of the top-10 nodes per centrality metric onto the
Italy basemap. The **convergence map** (colour = count of metrics for which a
node ranks in the top-10) identifies structurally robust hubs: cells that are
simultaneously active, well-connected, and located on critical fault bridges.

Seismological expectation: hubs should cluster along the Apennine arc and
in historically active regions (L'Aquila, Amatrice, Irpinia); divergence
between metrics reveals nodes that are locally active but globally peripheral.

In [ ]:
# Interactive map: dropdown to switch metric, top-10 nodes on Italy basemap.
plot_geo_top_n_interactive_hybrid(
    df_centrality, top_n=20,
    title="Abe-Suzuki Italy (10 km)",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
)

# Overlap map: colour = number of metrics for which a node is in the top-10.
# Multi-metric hubs (warm colours) identify the most structurally critical cells.
plot_geo_centrality_overlap_hybrid(
    df_centrality, top_n=10,
    title="Abe-Suzuki Italy (10 km)",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
)

## Community Detection

## Community Detection — Five Algorithms

The Abe-Suzuki network is a **directed transition network** (Markov chain on spatial
cells). Conceptually, communities are sets of cells that form quasi-closed seismic flow
basins — once a sequence enters a basin, it tends to stay there.  Two classes of method
are applied: **flow-based** (InfoMap, run on the directed weighted graph — conceptually
correct for this network type) and **structural** (directed Louvain via Leiden, plain
and consensus, finding densely connected groups under directed modularity).

**InfoMap** (Rosvall & Bergstrom 2008) is the primary method for directed transition
networks.  It minimises the map equation
$$L(\mathcal{M}) = q_\curvearrowleft H(\mathcal{Q}) + \sum_{i=1}^{m} p_i^\circlearrowleft H(\mathcal{P}^i),$$
which balances the cost of describing inter- vs intra-community movement in the directed
random walk on the **weighted** graph (link weights become transition probabilities).

**Directed Louvain** (Blondel *et al.* 2008, via Leiden algorithm — Traag *et al.* 2019)
maximises the Reichardt–Bornholdt directed modularity $Q_\gamma$ at resolution
$\gamma = 1.0$ (standard Newman-Girvan setting). Structural communities here correspond
to cells that frequently send *and* receive aftershock activity within the same group.

**Consensus Louvain** (Lancichinetti & Fortunato 2012): N Louvain runs at the **same** γ
→ co-occurrence matrix $C_{ij}$ → final Louvain on $C$; eliminates single-run noise but
does not change the granularity scale set by γ.

All three partitions are compared pairwise via NMI (3×3 matrix). Agreement between
InfoMap and directed Louvain is the key diagnostic: high NMI implies flow basins ≈
densely-connected structural clusters; low NMI implies directed flow organises the
network differently from modularity. **Both Louvain variants must use the same γ**,
otherwise their disagreement reflects scale difference, not method difference.

## DATA FILTERING FOR BETTER DETECTION

In [ ]:
# ================= DATABASE FILTERING ===================
df_net_test = df_net[
    (df_net["magnitude"] > 0)  #&
    #(df_net["time"].dt.year >= 2005) &
    #(df_net["time"].dt.year <= 2015)
].reset_index(drop=True)
print(f"Test filter: {len(df_net_test):,} earthquakes remain")


# ================= NETWORK CREATION ===================
G = build_abe_suzuki_network_custom_hybrid(
    df_net_test, 
    cell_size_km=30, 
    spatial_threshold_km = 300.0,   # filter on spatial distance between two consecutive EQ
    time_threshold_sec = 24 * 3600, # filter on time elapsed between two consecutive EQ (in seconds)
    target_crs=TARGET_CRS, 
    alpha = 0.7,            # magnitude scaling
    tau = 1 * 86400.0,      # temporal scale (seconds) ~ for exponential
    r0 = 50.0,              # spatial scale (km) ~ for exponential
    info=True
)

print(f"\n30 km hybrid network: {G.number_of_nodes():,} nodes  {G.number_of_edges():,} edges")

# ================= GIANT COMPONENT ===================
print("--- Giant Component ---")
wcc     = list(nx.weakly_connected_components(G)) # This finds connected components ignoring edge direction
G_giant = G.subgraph(max(wcc, key=len)).copy()    # This selects the largest connected component (in number of node
pct     = G_giant.number_of_nodes() / G.number_of_nodes() * 100
print(f"Components: {len(wcc)} | Giant: {G_giant.number_of_nodes():,} nodes ({pct:.1f}%)")

## DIRECTED LOUVAIN

**Directed Louvain** (via the Leiden algorithm — Traag *et al.* 2019,
using the Leicht–Newman 2008 directed modularity)
$$Q_d = \frac{1}{m}\sum_{ij}\left[A_{ij} - \frac{k_i^{\text{out}}k_j^{\text{in}}}{m}\right]\delta(c_i,c_j)$$
groups nodes that send *and* receive seismic activity within the same
community, going beyond edge density alone.  At resolution $\gamma = 1$
this is the standard Newman-Girvan modularity on the directed graph; the
Leiden optimiser is preferred over vanilla Louvain because it guarantees
internally well-connected communities.

In [ ]:
# ── DIRECTED LOUVAIN COMPUTATION ─────────────────────
# Single γ shared by plain and consensus Louvain — different γ would give
# partitions at different granularity scales and an artificially low NMI
# between the two methods.
LOUVAIN_RESOLUTION = 1.0

print(f"Louvain (directed, γ={LOUVAIN_RESOLUTION:.1f})")
community_map_directed = run_louvain_directed_hybrid(G_giant, resolution=LOUVAIN_RESOLUTION)
k_louvain = len(set(community_map_directed.values()))
print(f"  {k_louvain} communities")

plot_community_geo_hybrid(
    G_giant, community_map_directed,
    title="Abe-Suzuki Italy (10 km)",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH, bounds=_IT_BOUNDS,
    min_community_size=10, method_name="Louvain (directed)",
)

## CONSENSUS LOUVAIN

**Consensus Louvain** (Lancichinetti & Fortunato 2012) addresses the
well-known instability of single-run modularity optimisation: 100 (or
here 10 — adjust if precision matters) independent Louvain runs are
performed, the co-occurrence matrix $C_{ij}$ counts how often nodes
$i$ and $j$ end up in the same community, and a final Louvain run is
performed on $C$ thresholded at 0.5.  Communities that survive this
procedure are robust to algorithmic noise.

In [ ]:
# ── CONSENSUS DIRECTED LOUVAIN (HYBRID NETWORK) ─────────────────────

print(f"Louvain (consensus, directed, hybrid, γ={LOUVAIN_RESOLUTION:.1f})...")

# safety cleanup (recommended for hybrid weighted graphs)
G_input = G_giant.copy()
G_input.remove_edges_from(nx.selfloop_edges(G_input))

community_map_consensus = run_louvain_consensus_hybrid(
    G_input,
    n_runs=50,                       # 50 Louvain runs for the co-occurrence count
    resolution=LOUVAIN_RESOLUTION,   # MUST match plain Louvain γ above
    directed=True,
    threshold=0.5,                   # standard Lancichinetti-Fortunato cutoff
    max_iter=1,                      # single consensus pass (not iterative —
                                     # iterating loses directedness and over-fragments)
    sample_pairs=False,              # use ALL co-occurring pairs — random
                                     # subsampling drops genuine co-occurrences
                                     # below the 0.5 threshold for any
                                     # community larger than max_pairs_per_comm
)

k_louvain = len(set(community_map_consensus.values()))

print(f"  {k_louvain} communities (consensus hybrid)")

plot_community_geo_hybrid(
    G_giant, community_map_consensus,
    title="Abe-Suzuki Italy (10 km)",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH, bounds=_IT_BOUNDS,
    min_community_size=10, method_name="Louvain (consensus directed)",
)

## INFOMAP

**InfoMap** (Rosvall & Bergstrom 2008) is the primary flow-based method
for directed transition networks.  It minimises the map equation
$$L(\mathcal{M}) = q_\curvearrowleft H(\mathcal{Q})
+ \sum_{i=1}^{m} p_i^\circlearrowleft H(\mathcal{P}^i),$$
which balances the cost of describing inter-community vs intra-community
movement in the directed random walk.  Communities correspond to regions
where a random walker tends to remain trapped — for earthquake networks,
spatial sub-systems that share aftershock cascades among themselves.

In [ ]:
# ── InfoMap on the DIRECTED graph ──────
print("InfoMap (directed)...")
community_map_infomap = run_infomap_hybrid(G_giant, directed=True, seed=42)
k_infomap = len(set(community_map_infomap.values()))
print(f"Found {k_infomap} communities")

plot_community_geo_hybrid(
    G_giant, community_map_infomap,
    title="Abe-Suzuki Italy (10 km)",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH, bounds=_IT_BOUNDS,
    min_community_size=10, method_name="InfoMap",
)

## NMI

In [ ]:
print("Computing NMI across methods...")

partitions = {
    "Louvain (directed)": community_map_directed,
    "Consensus Louvain":  community_map_consensus,
    "InfoMap":            community_map_infomap
}

nmi_df = compute_nmi_matrix(partitions)

print(nmi_df)
plot_nmi_heatmap(nmi_df)

## NMI — Filtered (small communities removed)

The raw NMI above is dragged down by the **fragment tail** of Louvain at
$\gamma = 0.3$: of the 767 raw communities, only $\sim 12$ have $\geq 10$ cells
(the rest are singletons and dyads — micro-fragments below the modularity null
at this weight scale, see the discussion above). InfoMap, in contrast, finds
5 macro-flow-basins with no fragment tail.

When NMI compares these directly, each Louvain micro-fragment splits
arbitrarily across InfoMap's 5 large communities, **driving the score down for
reasons unrelated to method disagreement on the genuine structure**. The
visualizations already filter at `min_community_size=10`; the NMI comparison
should be consistent.

Below we re-compute NMI after dropping communities of size $< 10$ from every
partition — an apples-to-apples comparison of the **meaningful structure that
each method discovered**, decoupled from the noise floor.

In [ ]:
def filter_partition(part: dict, min_size: int = 10) -> dict:
    """Drop nodes whose community has fewer than ``min_size`` members."""
    counts = pd.Series(list(part.values())).value_counts()
    keep = set(counts[counts >= min_size].index)
    return {n: c for n, c in part.items() if c in keep}

partitions_clean = {
    name: filter_partition(p, min_size=10) for name, p in partitions.items()
}
print("Communities ≥ 10 cells per method:")
for name, p in partitions_clean.items():
    print(f"  {name:22s}  {len(set(p.values())):3d} communities, "
          f"{len(p):,} nodes retained (of {len(partitions[name]):,})")

nmi_df_clean = compute_nmi_matrix(partitions_clean)
print()
print(nmi_df_clean)
plot_nmi_heatmap(nmi_df_clean)

## HDBSCAN-geo — Spatial Null Baseline

HDBSCAN clustering on the **geographic coordinates of the cells only** — no
graph, no edge weights, no flow. This is a *spatial null baseline* that
answers the question:

> *Is the network adding community structure beyond what pure spatial
> proximity already explains?*

If NMI(Louvain, HDBSCAN-geo) $\approx 0.8$ → network methods are
rediscovering geography. If NMI $\approx 0.3$–$0.6$ → the network captures
structure beyond spatial clustering (triggering relationships, flow,
modularity at finer scales than density).

HDBSCAN (Campello *et al.* 2013; McInnes & Healy 2017) is a density-based
clusterer: it builds a hierarchy of clusters from variable-density mutual-
reachability distance and prunes by ``min_cluster_size``. Cells in
low-density regions are labelled as noise (``-1``); for fair NMI we re-label
each noise cell as a unique singleton so the ``≥ 10`` cell filter excludes
them without lumping all noise into one artificial community.

We add HDBSCAN-geo to the comparison **before MM-SBM** so the graph
methods (Louvain, Consensus, InfoMap) are first measured against the pure
spatial null in isolation; MM-SBM is then added in the next stage as a
separate model-based method.

In [ ]:
community_map_hdbscan = run_hdbscan_geo_hybrid(
    G_giant, min_cluster_size=10, target_crs=TARGET_CRS,
)
n_clusters_hdbscan = len(set(community_map_hdbscan.values()))
print(f"HDBSCAN-geo: {n_clusters_hdbscan} clusters before filter "
      f"({len(community_map_hdbscan):,} cells; singletons re-labelled from noise)")

plot_community_geo_hybrid(
    G_giant, community_map_hdbscan,
    title="Abe-Suzuki Italy (hybrid) — spatial baseline",
    center_lat=41.9, center_lon=12.5, zoom=0,
    height=MAP_HEIGHT, width=MAP_WIDTH, bounds=_IT_BOUNDS,
    min_community_size=10, method_name="HDBSCAN (geo, density)",
)

## NMI — Filtered, with HDBSCAN-geo (4-method, pre-MM-SBM)

Same as the filtered 3-method NMI above but adds the HDBSCAN-geo spatial
null as a fourth row/column. Read **down** the HDBSCAN column: each entry
is how much of that graph-based method's partition is recoverable from
pure geographic density alone. The complement to 1.0 is the "network-
specific" contribution of that method.

In [ ]:
partitions_with_hdbscan = dict(partitions_clean)
partitions_with_hdbscan["HDBSCAN (geo)"] = filter_partition(
    community_map_hdbscan, min_size=10
)
print("Communities ≥ 10 cells per method (4-method matrix, with spatial baseline):")
for name, p in partitions_with_hdbscan.items():
    print(f"  {name:22s}  {len(set(p.values())):3d} communities, "
          f"{len(p):,} nodes retained")

nmi_df_4_hdbscan = compute_nmi_matrix(partitions_with_hdbscan)
print()
print(nmi_df_4_hdbscan)
plot_nmi_heatmap(nmi_df_4_hdbscan)

## MM-SBM — Custom Variational EM (Airoldi 2008)

**Mixed-Membership Stochastic Blockmodel** (Airoldi, Blei, Fienberg & Xing
2008) is the prof-recommended model-based community detection for the hybrid
weighted directed network. Unlike Louvain/InfoMap (hard partitions, each node
in one community), MM-SBM assigns each cell a **probability vector**
$\pi_p \in \Delta^{K-1}$ over $K$ blocks. The model:

For each ordered pair $(p, q)$:
$$ z_{p \to q} \sim \mathrm{Mult}(\pi_p), \quad
z_{q \to p} \sim \mathrm{Mult}(\pi_q), \quad
A_{pq} \sim \mathrm{Bernoulli}\!\big(B[z_{p\to q}, z_{q\to p}]\big). $$

Variational EM (Airoldi 2008 §3, eqs. 3, 5, 7) alternates a per-pair
multinomial update (E-step) with the closed-form $B$ and $\gamma$ updates
(M-step). Implemented in pure numpy in `src/community_custom.py` with full
$(N, N, K)$ tensor materialisation; runtime $\sim$ few minutes on the hybrid
giant at $K \sim 12$.

**Weight handling**: we use ``weight_mode='poisson'`` with a
$\log(1+w)$ transform of the hybrid edge weights. The hybrid weights span
~6 decades (min ≈ 0.02, max ≈ 4.8e4), so a raw Poisson likelihood is
numerically unstable and integer rounding would collapse the small-weight
tail (median weight ≈ 0.02 rounds to 0). The $\log1p$ transform compresses
the range to ~0–11 while preserving relative ordering, giving stable
Poisson rates.

**Initialisation**: ``init='louvain'`` seeds the EM with the directed
Louvain partition (γ=1, RB, the same used above). Pure random
initialisation traps the EM in a degenerate fixed point where every block
is interchangeable; Louvain breaks the symmetry.

In [ ]:
MMSBM_K = max(2, k_louvain)  # K = number of Louvain communities ≥ 10 (12 typical)
print(f"Running MM-SBM-EM with K={MMSBM_K}, n_iter=50, init='louvain'...")
pi_mmsbm, community_map_mmsbm = run_mmsbm_custom_hybrid(
    G_giant,
    K=MMSBM_K,
    n_iter=50,
    init="louvain",
    weight_mode="poisson",
    seed=42,
    verbose=True,
)
n_blocks_mmsbm = len(set(community_map_mmsbm.values()))
print(f"\nMM-SBM: π matrix shape={pi_mmsbm.shape}, "
      f"{n_blocks_mmsbm} non-empty blocks (of K={MMSBM_K} possible)")

# Per-cell entropy of the membership vector — bright cells (high entropy) are
# the most overlapping ones (likely fault intersections per Yang & Leskovec 2013)
mmsbm_entropy = -np.sum(pi_mmsbm * np.log(pi_mmsbm + 1e-10), axis=1)
print(f"  membership entropy: mean={mmsbm_entropy.mean():.3f}, "
      f"max={mmsbm_entropy.max():.3f} (max possible = log K = {np.log(MMSBM_K):.3f})")

plot_community_geo_hybrid(
    G_giant, community_map_mmsbm,
    title="Abe-Suzuki Italy (10 km hybrid)",
    center_lat=41.9, center_lon=12.5, zoom=0,
    height=MAP_HEIGHT, width=MAP_WIDTH, bounds=_IT_BOUNDS,
    min_community_size=10, method_name="MM-SBM (custom EM, argmax)",
)

## NMI — Filtered, with MM-SBM (5-method, final)

Adding the custom MM-SBM (Louvain-initialised) on top of the 4-method
HDBSCAN-augmented matrix → final **5-method NMI**.

**Important caveat for interpretation**: the custom MMSB was initialised
from the Louvain partition to break the degenerate EM fixed point, so its
NMI vs Louvain is **biased high by construction** — the EM converges in
Louvain's basin of attraction. The genuinely independent comparisons for
MM-SBM are vs **InfoMap** (flow-based, totally different objective) and
vs **HDBSCAN-geo** (purely spatial, no graph at all).

In [ ]:
partitions_final = dict(partitions_with_hdbscan)  # 4-method incl. HDBSCAN
partitions_final["MM-SBM (custom)"] = filter_partition(community_map_mmsbm, min_size=10)
print("Communities ≥ 10 cells per method (5-method matrix):")
for name, p in partitions_final.items():
    print(f"  {name:22s}  {len(set(p.values())):3d} communities, "
          f"{len(p):,} nodes retained")

nmi_df_5 = compute_nmi_matrix(partitions_final)
print()
print(nmi_df_5)
plot_nmi_heatmap(nmi_df_5)

## GRID SEARCH

In [ ]:
G = build_abe_suzuki_network_custom_hybrid(
    df_net, 
    cell_size_km=20, 
    spatial_threshold_km = 300.0,   # filter on spatial distance between two consecutive EQ
    time_threshold_sec = 24 * 3600, # filter on time elapsed between two consecutive EQ (in seconds)
    target_crs=TARGET_CRS, 
    alpha = 0.7,            # magnitude scaling
    tau = 1 * 86400.0,      # temporal scale (seconds) ~ for exponential
    r0 = 20.0,              # spatial scale (km) ~ for exponential
    info=True
)

from joblib import Parallel, delayed
import itertools

# ── parameter ranges ────────────────────────────────────────────────────────
range_cell_size_km = [30, 35, 40, 45, 50]
range_alpha = [1.1, 1.2, 1.3, 1.4, 1.5]
range_r0 = [10, 20, 30, 40]
range_tau_days = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

TARGET_CRS = "epsg:32632"
RESOLUTION = 1.0

df_net_test = df_net[df_net["magnitude"] > 1.5].reset_index(drop=True)

param_grid = list(itertools.product(
    range_cell_size_km,
    range_alpha,
    range_r0,
    range_tau_days
))

def evaluate_params(cell_size_km, alpha, r0, tau_days):

    tau_sec = tau_days * 86400.0

    # ── build HYBRID network with correct params ────────────────────────────
    G = build_abe_suzuki_network_custom_hybrid(
        df_net_test,
        cell_size_km=cell_size_km,
        spatial_threshold_km=300.0,
        time_threshold_sec=24 * 3600,
        target_crs=TARGET_CRS,
        alpha=alpha,
        tau=tau_sec,
        r0=r0,
        info=False
    )

    # ── community detection ────────────────────────────────────────────────
    comm_louvain = run_louvain_directed_hybrid(G, resolution=RESOLUTION)
    comm_infomap = run_infomap_hybrid(G, directed=True, seed=42)

    # ── NMI (alignment fixed version assumed) ───────────────────────────────
    nmi = compute_nmi(comm_louvain, comm_infomap)

    return (cell_size_km, alpha, r0, tau_days, nmi)


results = Parallel(n_jobs=-1, verbose=10)(
    delayed(evaluate_params)(cs, a, r, t)
    for cs, a, r, t in param_grid
)

df_results = pd.DataFrame(
    results,
    columns=["cell_size_km", "alpha", "r0", "tau_days", "nmi"]
)

df_results.sort_values("nmi", ascending=False).head(10)

## Assortativity

In complex network science, **degree assortativity** describes the mixing patterns
of a system — whether nodes preferentially connect to others with a similar degree.
In an assortative network ($\mu > 0$), highly connected hubs cluster together,
creating a resilient core. In a disassortative network ($\mu < 0$), hubs connect
preferentially to low-degree peripheral nodes.

When analyzing networks with heavy-tailed degree distributions ($\gamma < 2$),
diagnosing true physical assortativity is complicated by **finite-size effects**
arising from two distinct mathematical thresholds:

$$k_{\mathrm{nat}} \sim N^{1/(\gamma-1)}, \qquad k_{\mathrm{str}} = \sqrt{2L} = \sqrt{\langle k \rangle N}$$

**Structural disassortativity** occurs automatically when $k_{\mathrm{nat}} > k_{\mathrm{str}}$.
When the degree distribution expects micro-hubs larger than $k_{\mathrm{str}}$,
these hubs physically run out of other hubs to connect to without creating
forbidden multi-edges — forcing connections to smaller peripheral cells instead.

**Complete pipeline**: (1) compute raw $k_{nn}(k)$ on the simple undirected graph;
(2) apply logarithmic binning to smooth the sparse high-$k$ tail; (3) fit the
mixing exponent $\mu$ strictly to the intrinsic regime $k < k_{\mathrm{str}}$;
(4) plot results on log-log axes with the structural cutoff marked.

In [ ]:
raw_data, binned_data, metrics = analyze_degree_correlations_hybrid(
    G=G_10km,
    gamma=gamma_10km,
    num_bins=15,
    weighted = True,
    save=True,
)

## Assortativity — Randomization Test

**Degree-preserving randomization** (configuration null model): $L \times 10$
edge swaps ($A$-$B$ and $C$-$D \to A$-$D$ and $C$-$B$) scramble topology while
keeping the exact degree sequence of every node intact.  This destroys all
spatial-temporal correlations but preserves the degree distribution.

Comparing the original and rewired $k_{nn}(k)$ curves below $k_{\mathrm{str}}$
directly separates structural artifacts (same in both) from genuine seismic
coupling patterns (different in original only).
If $|\mu_{\mathrm{orig}} - \mu_{\mathrm{rand}}| < 0.03$, the network is
intrinsically neutral: seismic cells trigger consecutive earthquakes
independently of their activity level (Pastor-Satorras *et al.* 2001).

In [ ]:
run_binned_randomization_test_hybrid(
    G=G_10km,
    gamma=1.84,
    num_bins=15,
    n_swaps_per_edge=10,
    use_weighted = False,
    save=True,
)

## DA APPROFONDIRE IL DISCORSO RANDOMIZATION